# 📷 P1 · Daten sammeln

**Projekt-Notebook 1 von 3** · läuft **lokal** auf deinem Laptop

```
 ▶ P1 Daten sammeln  →  P2 Labeln & prüfen  →  P3 Training in Colab
   ─────────────────
   Steckbrief · Regeln · Fotos · Import · Sichten · Katalog
```

In diesem Notebook legst du fest, **was** dein Modell erkennen soll, und sammelst die **Bilder** dafür.
Am Ende liegen in `2_bilder/` geprüfte, richtig gedrehte Bilder, bereit zum Labeln.

| Abschnitt | Was du tust |
|---|---|
| 1 · Start | Werkzeuge laden, Ordner anlegen |
| 2 · Steckbrief | Ziel, Klassen und Einsatzort festlegen |
| 3 · Regeln | Wie gute Trainingsbilder aussehen, mit Aufnahmeplan |
| 4 · Fotos aufnehmen | mit dem Handy oder der Webcam |
| 5 · Importieren | drehen, verkleinern, umbenennen (automatisch) |
| 6 · Sichten | durchblättern, drehen, aussortieren |
| 7 · Katalog | Qualität prüfen: dunkel? unscharf? doppelt? |
| 8 · Bilanz | Reicht die Menge? Checkliste |

> 📖 Theorie zu diesem Notebook: [Gute Trainingsbilder](https://rdf-sul.github.io/YOLO_Training/projekt/bilder-aufnehmen/) ·
> [P1 auf der Lernseite](https://rdf-sul.github.io/YOLO_Training/projekt/p1-daten-sammeln/)
>
> ▶ Führe die Zellen **von oben nach unten** aus: ▶-Knopf oder <kbd>Shift</kbd>+<kbd>Enter</kbd>.
> Du kannst jede Zelle **beliebig oft** wiederholen.

---
## 1 · Start

Diese Zelle lädt die Werkzeuge aus `werkzeuge.py` und legt die Projektordner an:

```
yolo-projekt/
├── 1_rohbilder/            ← hier kommen deine Fotos hinein (Unterordner pro Person/Kamera)
├── 2_bilder/               ← vorbereitete Bilder (erzeugt in Abschnitt 5)
├── 3_labelstudio_export/   ← Exporte aus Label Studio (P2)
├── 4_datensatz/            ← fertiger Datensatz für das Training (P2)
├── P1_daten_sammeln.ipynb
├── P2_labeln_und_pruefen.ipynb
├── P3_training_colab.ipynb
└── werkzeuge.py
```

In [ ]:
%matplotlib inline
import werkzeuge as wz
import pandas as pd

wz.ordner_anlegen()
print("✅ Werkzeuge geladen, Ordner bereit.")
print("   HEIC-Fotos (iPhone) lesbar:", "ja" if wz.HEIC_OK else "nein – uv add pillow-heif")

---
## 2 · Projekt-Steckbrief

Bevor du ein einziges Foto machst, beantworte diese Fragen **schriftlich**. Ein unklares Ziel ist der häufigste
Grund für ein schlechtes Modell, noch vor zu wenigen Bildern.

| Frage | Beispiel |
|---|---|
| **Was** soll erkannt werden? | Bauteile auf dem Arbeitsplatz: LED, Taster, Widerstand |
| **Wo** wird das Modell eingesetzt? | am Laborplatz, Webcam von schräg oben, Neonlicht |
| **Welche Kamera** wird im Einsatz verwendet? | Laptop-Webcam (1280×720) |
| **Woran unterscheidet ein Mensch** die Klassen? | Form und Farbringe … |
| Was zählt **nicht** zu einer Klasse? | lose Drähte, Kondensatoren → kein Label |

**Regeln für Klassen**

- Klassen müssen **sichtbar unterscheidbar** sein. Was du auf dem Foto nicht auseinanderhalten kannst, kann das Modell auch nicht.
- **Lieber wenige Klassen** (2–4) gut als viele schlecht. Jede Klasse kostet Bilder und Zeit.
- Lege für jede Klasse eine **Definition** fest. Beispiel: „Widerstand: Box nur um den Körper **mit** Anschlussdrähten.“
  Alle in der Gruppe halten sich daran.
- Klassennamen: **ohne Leerzeichen und Umlaute**, z. B. `Widerstand`, `led_rot`.
- Lege ein **Erfolgskriterium** fest: Woran merkt ihr, dass das Modell gut genug ist? Ohne Ziel weiß niemand, wann das Projekt fertig ist.

✏️ **Trage deinen Steckbrief in die nächste Zelle ein und führe sie aus.** Gespeichert wird er in `projekt.yaml`,
P2 und P3 lesen ihn dort.

In [ ]:
PROJEKT = {
    "name": "Bauteile erkennen",                     # ← kurzer Projektname
    "ziel": "Erkennt LEDs, Taster und Widerstände auf dem Laborplatz.",
    "klassen": ["LED", "Taster", "Widerstand"],      # ← deine Klassen (Reihenfolge = Klassennummer 0, 1, 2 …)
    "definitionen": {                                # ← Wie genau wird gelabelt? (für die ganze Gruppe gleich!)
        "LED": "LED-Körper und beide Beinchen",
        "Taster": "ganzer Taster inkl. Anschlüsse",
        "Widerstand": "Körper mit Anschlussdrähten",
    },
    "einsatz": "Laborplatz, Webcam schräg von oben, Neonlicht",
    "kamera_im_einsatz": "Webcam",                   # Webcam / Handy / Industriekamera …
    "ziel_objekte_pro_klasse": 200,                  # Richtwert, siehe Abschnitt 3
    "erfolgskriterium": "Jede Klasse mAP50 ≥ 0,8 auf val UND im Webcam-Test 9 von 10 Objekten erkannt",
    "gruppe": ["Anna", "Ben"],                       # ← wer arbeitet mit?
}

wz.projekt_speichern(PROJEKT)
print("✅ Steckbrief gespeichert in projekt.yaml")
print("   Klassen:", ", ".join(f"{i} = {k}" for i, k in enumerate(PROJEKT["klassen"])))

---
## 3 · Regeln für gute Trainingsbilder

Ein YOLO-Modell lernt **nur** aus Beispielen. Was in den Trainingsbildern nicht vorkommt, kann es später nicht.
Die wichtigste Regel lautet deshalb:

> 🎯 **Die Trainingsbilder müssen so aussehen wie die Bilder im späteren Einsatz – und zwar in ihrer ganzen Vielfalt.**

### 3.1 Wie viele?

Gezählt werden **Objekte** (Boxen), nicht Bilder. Ein Foto mit 4 Widerständen liefert 4 Beispiele.

| Stufe | Objekte **pro Klasse** | Wofür |
|---|---|---|
| 🟡 Prototyp | ab ca. **100** | einfache, gut unterscheidbare Objekte, feste Umgebung |
| 🟢 **Solides Projekt** | **200–300** | mehrere Hintergründe und Lichtsituationen, **unser Ziel** |
| 🔵 Profi (Empfehlung von Ultralytics) | ≥ 1.500 Bilder / ≥ 10.000 Objekte | Industrie, wechselnde Bedingungen |

Dazu kommen **5–10 % Hintergrundbilder**: Fotos **ohne** deine Objekte, aber mit typischen Störern
(z. B. Kondensatoren, Werkzeug). Sie senken die Zahl der Fehlalarme.

Alle Klassen sollten **ungefähr gleich oft** vorkommen. Sonst lernt das Modell die seltene Klasse schlecht.

### 3.2 Welche Kamera? Handy oder Webcam?

| | 📱 Handy | 💻 Webcam |
|---|---|---|
| Bildqualität | sehr gut, scharf, viele Megapixel | mittel, oft rauschig, Weitwinkel |
| Bequemlichkeit | viele Fotos schnell, überall | direkt am Rechner, Serienaufnahme |
| Fallen | **EXIF-Drehung**, HEIC-Format, Porträt-/Beauty-Modus | Autofokus pumpt, Bewegungsunschärfe |

**Regel:** Fotografiere **überwiegend mit der Kamera, die später im Einsatz ist** (siehe Steckbrief).
Ein Modell, das nur scharfe Handyfotos kennt, scheitert oft an einer rauschigen Webcam.
Ein **Mix** aus mehreren Kameras macht das Modell robuster. Mindestens die Hälfte sollte aber von der Einsatzkamera stammen.

📱 **Einstellungen am Handy:** normaler Fotomodus (kein Porträt, kein Beauty-Filter, **kein Digitalzoom**),
Blitz aus, Live-Foto aus. HEIC ist kein Problem, Notebook-Abschnitt 5 wandelt es um.

### 3.3 Vielfalt: Was soll sich zwischen den Fotos ändern?

| Merkmal | ✅ so | ❌ nicht so |
|---|---|---|
| **Hintergrund** | mehrere Unterlagen: Tisch, Matte, Papier, Platine | immer dieselbe weiße Unterlage |
| **Licht** | Tageslicht, Deckenlicht, Schreibtischlampe, etwas Schatten | nur ein Licht, Gegenlicht, Blitz |
| **Abstand** | nah, mittel, weit (wie im Einsatz) | Objekt füllt immer das halbe Bild |
| **Lage** | gedreht, schräg, liegend, umgedreht | alle Teile exakt ausgerichtet |
| **Anzahl** | 1 bis viele Objekte pro Bild, gemischte Klassen | immer genau ein Objekt in der Mitte |
| **Verdeckung** | Objekte berühren sich, teils verdeckt, angeschnitten am Rand | nur frei liegende Einzelteile |
| **Exemplare** | verschiedene Farben, Größen, Hersteller | immer dasselbe Bauteil |

⚠️ **Die „Kluger-Hans“-Falle:** Liegen alle LEDs auf weißem Papier und alle Widerstände auf der grünen Matte,
lernt das Modell **„weiß = LED“** statt der Form. Achte darauf, dass **jede Klasse auf jedem Hintergrund** vorkommt.

### 3.4 Technische Qualität

- **Größe der Objekte:** Ein Objekt sollte mindestens etwa **1/20 der Bildbreite** groß sein.
  Kleinere Objekte verschwinden fast, wenn das Bild für das Training auf 640 px verkleinert wird.
- **Scharf** und ruhig halten. Bei der Webcam das Objekt kurz liegen lassen, bis der Autofokus sitzt.
- **Keine Fast-Duplikate:** 30 Serienfotos derselben Szene bringen kaum mehr als 3.
  Zwischen zwei Fotos **etwas verändern** (Lage, Abstand, Licht …). Das gilt auch für die Serienaufnahme der Webcam:
  Sie ist nur sinnvoll, wenn du die Objekte **zwischen** den Auslösungen bewegst.
- **Datenschutz:** keine Gesichter, Namen oder Bildschirme mit persönlichen Daten aufs Foto.

### 3.5 Aufnahmeplan

Plane deine Fotos wie eine **Messreihe**. Die folgende Zelle berechnet aus deinem Steckbrief,
wie viele Fotos du ungefähr brauchst, und verteilt sie auf die Kombinationen aus Hintergrund und Licht.
Trage deine eigenen Hintergründe und Lichtsituationen ein.

In [ ]:
HINTERGRUENDE = ["weißes Papier", "grüne Schneidematte", "Holztisch", "Lochrasterplatine"]   # ← anpassen
LICHT         = ["Deckenlicht", "Tageslicht am Fenster", "Schreibtischlampe"]               # ← anpassen
OBJEKTE_PRO_FOTO = 4      # ← Schätzung: Wie viele Objekte sind im Schnitt auf einem Foto?

klassen = PROJEKT["klassen"]
ziel = PROJEKT["ziel_objekte_pro_klasse"]
fotos = ziel * len(klassen) / OBJEKTE_PRO_FOTO
hintergrund_fotos = round(fotos * 0.08)
kombis = len(HINTERGRUENDE) * len(LICHT)
pro_kombi = -(-round(fotos) // kombis)  # aufrunden

print(f"Ziel: {ziel} Objekte × {len(klassen)} Klassen ÷ {OBJEKTE_PRO_FOTO} Objekte pro Foto "
      f"≈ {fotos:.0f} Fotos  +  ca. {hintergrund_fotos} Hintergrundfotos (ohne Objekte)\n")
plan = pd.DataFrame(pro_kombi, index=HINTERGRUENDE, columns=LICHT)
plan.index.name = "Hintergrund ↓ / Licht →"
print(f"📋 Aufnahmeplan: je ca. {pro_kombi} Fotos pro Kombination (Abstand und Lage dabei variieren!)")
display(plan)
if PROJEKT.get("gruppe"):
    print(f"\n👥 Aufgeteilt auf {len(PROJEKT['gruppe'])} Personen: je ca. {fotos / len(PROJEKT['gruppe']):.0f} Fotos")

---
## 4 · Fotos aufnehmen

Alle Fotos kommen nach **`1_rohbilder/`**, und zwar in einen **eigenen Unterordner pro Person und Kamera**,
z. B. `1_rohbilder/anna_handy/` oder `1_rohbilder/ben_webcam/`.
Der Ordnername wird Teil des Bildnamens. So weißt du später immer, woher ein Bild stammt.
Verwende dabei das Wort aus `kamera_im_einsatz` (z. B. `_webcam`), dann kann P1 am Ende prüfen, wie viele Fotos
von der Einsatzkamera stammen.

### 4a · Mit dem Handy

1. Fotos nach dem Aufnahmeplan machen.
2. Auf den Laptop übertragen: USB-Kabel, OneDrive/Teams oder AirDrop.
   **Nicht** per Messenger verschicken, der verkleinert und komprimiert die Bilder.
3. Dateien **unverändert** in `1_rohbilder/<name>_handy/` kopieren. Nichts vorher drehen oder bearbeiten!

### 4b · Mit der Webcam

Die nächste Zelle öffnet ein **Kamerafenster**:

| Taste | Wirkung |
|---|---|
| <kbd>Leertaste</kbd> | ein Foto |
| <kbd>S</kbd> | Serienaufnahme an/aus (jede Sekunde ein Foto): Objekte zwischendurch bewegen! |
| <kbd>Q</kbd> oder <kbd>Esc</kbd> | beenden |

Erscheint kein Bild, versuche `KAMERA = 1`. Oder schließe Programme, die die Kamera benutzen (Teams, Zoom …).

In [ ]:
KAMERA_ORDNER = "ben_webcam"   # ← Name des Unterordners in 1_rohbilder/
KAMERA = 0                     # 0 = eingebaute Kamera, 1 = zweite/USB-Kamera

wz.webcam_aufnahme(KAMERA_ORDNER, kamera=KAMERA, serie_sekunden=1.0)

---
## 5 · Importieren: drehen, verkleinern, umbenennen

Die nächste Zelle holt **alle** Fotos aus `1_rohbilder/` nach `2_bilder/` und erledigt dabei drei Dinge:

1. **Richtig herum drehen.** Handys speichern Hochkant-Fotos oft quer und vermerken nur in den Metadaten (EXIF):
   „bitte um 90° drehen“. Viele Programme beachten das, manche nicht. Dann passen Bild und Boxen später nicht zusammen.
   Darum wird die Drehung hier **fest ins Bild übernommen** und die Metadaten werden entfernt.
2. **Verkleinern** auf höchstens 1280 px (längste Seite). Das Training arbeitet ohnehin mit 640 px,
   12-Megapixel-Fotos würden nur Speicher und Zeit kosten.
3. **Umbenennen** in `<unterordner>_<originalname>.jpg`. Eindeutig, ohne Leer- und Sonderzeichen. HEIC/PNG werden zu JPG.

Die Originale in `1_rohbilder/` bleiben **unverändert**. Wenn neue Fotos dazukommen, führst du die Zelle einfach
noch einmal aus. Schon importierte Bilder werden übersprungen.

In [ ]:
MAX_SEITE = 1280   # längste Bildseite in Pixeln (1280 ist ein guter Wert, auch für spätere Experimente)

import_tabelle = wz.bilder_importieren(max_seite=MAX_SEITE)
if len(import_tabelle):
    display(import_tabelle[import_tabelle.status != "schon importiert"].head(15))

---
## 6 · Sichten

Schau dir **jedes** Bild einmal an. Du bist die Qualitätskontrolle!

- **Vorschaubilder:** viele kleine Bilder auf einen Blick
- **Bildbrowser:** einzeln blättern, mit Knöpfen
  - **⟲ / ⟳** dreht ein Bild, das falsch herum liegt (dauerhaft).
  - **🗑 aussortieren** verschiebt ein Bild nach `2_bilder/_aussortiert/`. Es wird nicht gelöscht und lässt sich zurückholen.

Aussortieren solltest du: verwackelte Bilder, Bilder ohne erkennbares Objekt (außer gewollte Hintergrundbilder),
Bilder mit Personen oder privaten Daten, versehentliche Doppelte.

⚠️ **Drehen nur jetzt, vor dem Labeln!** Wird ein Bild nach dem Labeln gedreht, passen die Boxen nicht mehr.
Sobald in `3_labelstudio_export/` Exporte liegen, sind die Drehknöpfe deshalb gesperrt.

In [ ]:
alle_bilder = wz.bilder_in(wz.BILDER)
print(f"{len(alle_bilder)} Bilder in 2_bilder/ (die ersten 30):")
wz.galerie(alle_bilder[:30], spalten=6)

In [ ]:
wz.bildbrowser()

---
## 7 · Katalog und Qualitätsprüfung

Die nächste Zelle misst jedes Bild und schreibt alles in die Tabelle **`katalog.csv`** (lässt sich in Excel öffnen):

| Spalte | Bedeutung | auffällig, wenn … |
|---|---|---|
| `quelle` | Unterordner aus `1_rohbilder/` (Person/Kamera) | – |
| `format` | hoch / quer | – |
| `helligkeit` | mittlerer Grauwert 0 (schwarz) … 255 (weiß) | < 50 oder > 210 |
| `kontrast` | Streuung der Grauwerte | < 8 (fast einfarbig) |
| `schaerfe` | Varianz des Laplace-Filters: Wie viele scharfe Kanten hat das Bild? | < 20 → vermutlich unscharf |
| `fast_gleich_wie` | Bild hat fast denselben „Fingerabdruck“ (Hash) wie ein anderes | gefunden |

💡 Das sind **Hinweise**, keine Urteile. Ein Bild einer weißen Wand ist „kontrastarm“, als Hintergrundbild aber vielleicht gewollt.
Entscheide selbst!

✏️ **Beobachtungsaufgaben**

1. Öffne `katalog.csv` in Excel und sortiere nach `schaerfe`. Sieh dir die 3 Bilder mit dem kleinsten und dem größten Wert an.
   Passt die Zahl zu deinem Eindruck?
2. Mach mit der Webcam absichtlich 5 Serienfotos, **ohne** etwas zu verändern, importiere sie und erstelle den Katalog neu.
   Findet die Spalte `fast_gleich_wie` sie? (Danach aussortieren!)

In [ ]:
katalog = wz.katalog_erstellen()
wz.uebersicht(katalog)

In [ ]:
auffaellig = katalog[katalog.hinweis != ""]
print(f"{len(auffaellig)} auffällige Bilder:")
display(auffaellig[["bild", "quelle", "helligkeit", "schaerfe", "fast_gleich_wie", "hinweis"]])

# auffällige Bilder ansehen und ggf. aussortieren:
dateien = [wz.BILDER / n for n in auffaellig.bild]
dateien = [d if d.exists() else next(wz.BILDER.rglob(d.name), d) for d in dateien]
wz.bildbrowser(dateien, texte=dict(zip(auffaellig.bild, auffaellig.hinweis)))

💡 **Grenzwerte anpassen:** Sind dir die Hinweise zu streng oder zu locker? Ändere z. B.
`wz.GRENZEN["unscharf"] = 40` und führe die Katalog-Zelle noch einmal aus.

---
## 8 · Bilanz: Reicht das?

Ob du **genug Objekte pro Klasse** hast, siehst du erst nach dem Labeln (P2).
Jetzt kannst du schon prüfen, ob die Zahl der **Bilder** zu deinem Aufnahmeplan passt und ob **alle Quellen** vertreten sind.

In [ ]:
katalog = wz.katalog_erstellen(speichern=True)
bilder_jetzt = len(katalog)
print(f"Bilder:   {bilder_jetzt}  von geplant ca. {fotos + hintergrund_fotos:.0f}  "
      f"→ {bilder_jetzt / (fotos + hintergrund_fotos):.0%}")
print("Quellen:  " + ", ".join(f"{q} ({n})" for q, n in katalog.quelle.value_counts().items()))
kamera = PROJEKT.get("kamera_im_einsatz", "").lower()
anteil = katalog.quelle.str.lower().str.contains(kamera[:4]).mean() if kamera else 0
print(f"Anteil von der Einsatzkamera ({PROJEKT.get('kamera_im_einsatz')}, erkannt am Ordnernamen): {anteil:.0%}"
      + ("  ✅" if anteil >= 0.5 else "  ⚠️ unter 50 % – mehr Fotos mit der Einsatzkamera machen!"))

### ✅ Checkliste vor P2

- [ ] Steckbrief ausgefüllt, **Klassen-Definitionen** mit der Gruppe abgesprochen
- [ ] Fotos nach **Aufnahmeplan**: verschiedene Hintergründe, Lichtsituationen, Abstände, Lagen
- [ ] Mindestens die Hälfte der Fotos mit der **Einsatzkamera**
- [ ] Einige **Hintergrundbilder** ohne Objekte dabei
- [ ] Alle Bilder **gesichtet**, falsch gedrehte korrigiert, schlechte aussortiert
- [ ] Katalog geprüft: keine ungewollten Doppelten, nichts zu dunkel oder unscharf

➡️ **Weiter mit `P2_labeln_und_pruefen.ipynb`.** Ab jetzt die Bilder in `2_bilder/` **nicht mehr drehen oder umbenennen.**

💡 Neue Fotos kannst du später jederzeit hinzufügen: in `1_rohbilder/` legen und Abschnitt 5–7 wiederholen.
Mehr dazu auf der Seite [Nachtraining](https://rdf-sul.github.io/YOLO_Training/projekt/nachtraining/).